# Screening Capacity: Gains Table, NHANES 2013-2014

**Author:** Ziang Zhang

A care-management team can only call a fixed number of people. Given a capacity of 5%, 10%, 20% or 30% of adults, contacted in order of predicted risk, how many CHD cases does the team reach?

No new model is fit. The out-of-fold predictions of the class-weighted logistic regression (the AUC 0.859 model of `nhanes_chd_audit_and_revalidation.ipynb`) are regenerated with the same code, folds and seed, and checked against `outputs/auc_bootstrap_ci.csv` before the table is computed. A gains table depends only on the ranking of predicted risk, so class weighting does not matter here.

**Population:** the analytic sample (5,751 adults with a CHD answer, 232 cases, 4.03%), unweighted. Numbers "per 10,000" are scaled from this sample and its 4.03% prevalence; they are **not** survey-weighted estimates for U.S. adults (weighted prevalence is 3.7%).

**Limitation:** NHANES is cross-sectional and CHD is a self-reported past diagnosis, so the model finds people who already have CHD. This demonstrates a capacity-based decision framework; it is not a member-screening model ready for deployment.

**Output:** `outputs/gains_table.csv`.

In [1]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Works when run from the repo root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs"
SEED = 42
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [2]:
# Same cohort, predictors, preprocessing and folds as the audit notebook (sections 2 and 5)
raw = pd.read_csv(DATA_DIR / "selected_features_v1.csv").rename(columns={"Blood-Rel Stroke": "StrokeHistory"})
labeled = raw[raw["disease_status"].notna()]
d = labeled.drop_duplicates("SEQN").reset_index(drop=True)
assert d["SEQN"].is_unique

categorical = ["Gender", "Vigorous Work", "Moderate Work", "Diabetes", "Blood-Rel Diabetes", "StrokeHistory"]
numeric = ["Age", "Systolic", "Diastolic", "Platelet Count", "Red-Cell Distribution Width", "Cholesterol",
           "Creatinine", "Glucose", "LDH", "Uric Acid", "Triglycerides", "HDL", "Glycohemoglobin"]
X = d[categorical + numeric].copy()
X[categorical] = X[categorical].astype(str)
y = (d["disease_status"] == "diseased").astype(int).to_numpy()

def make_preprocess():
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="mean")), ("scale", StandardScaler())]), numeric),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical),
    ])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(cv.split(X, y))
test_idx_all = np.concatenate([te for _, te in folds])
assert len(test_idx_all) == len(y) and len(np.unique(test_idx_all)) == len(y), "each participant must be tested exactly once"
assert all(len(np.intersect1d(tr, te)) == 0 for tr, te in folds)
print(f"{len(y):,} adults, {y.sum()} CHD cases ({y.mean():.2%}), {len(categorical) + len(numeric)} predictors")

5,751 adults, 232 CHD cases (4.03%), 19 predictors


In [3]:
def bootstrap_auc(y_true, prob, n_boot=1000, seed=SEED):
    """Same function and seed as the audit notebook, so resamples are paired across models."""
    r = np.random.default_rng(seed)
    n = len(y_true)
    aucs = []
    for _ in range(n_boot):
        idx = r.integers(0, n, n)
        if y_true[idx].sum() == 0:
            continue
        aucs.append(roc_auc_score(y_true[idx], prob[idx]))
    aucs = np.array(aucs)
    return aucs.mean(), np.percentile(aucs, 2.5), np.percentile(aucs, 97.5), aucs

# Class-weighted logistic regression, out-of-fold (audit notebook section 5)
lr_oof = np.zeros(len(y))
for tr, te in folds:
    pipe = Pipeline([("prep", make_preprocess()), ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))])
    pipe.fit(X.iloc[tr], y[tr])
    lr_oof[te] = pipe.predict_proba(X.iloc[te])[:, 1]

_, lr_lo, lr_hi, lr_boot = bootstrap_auc(y, lr_oof)
lr_auc = roc_auc_score(y, lr_oof)
ref = pd.read_csv(OUT_DIR / "auc_bootstrap_ci.csv").set_index("Model").loc["Logistic regression"]
assert (round(lr_auc, 3), round(lr_lo, 3), round(lr_hi, 3)) == (ref["AUC"], ref["CI_low"], ref["CI_high"]), \
    "LR out-of-fold predictions do not reproduce outputs/auc_bootstrap_ci.csv"
print(f"Logistic regression (class-weighted): AUC {lr_auc:.3f} ({lr_lo:.3f}-{lr_hi:.3f}), matches outputs/auc_bootstrap_ci.csv")

Logistic regression (class-weighted): AUC 0.859 (0.837-0.881), matches outputs/auc_bootstrap_ci.csv


## Gains table

Adults are ranked by predicted risk (highest first, ties broken by row order) and the top share is contacted; the number contacted is rounded up. Captured = share of all CHD cases among those contacted; PPV = share of contacted adults who have CHD; lift = PPV divided by the sample prevalence, i.e. how many times more cases than contacting the same number of adults at random.

In [4]:
order = np.argsort(-lr_oof, kind="mergesort")
y_ranked = y[order]
n, cases, prevalence = len(y), int(y.sum()), y.mean()

rows = []
for share in [0.05, 0.10, 0.20, 0.30]:
    k = int(np.ceil(share * n))
    found = int(y_ranked[:k].sum())
    rows.append({
        "CapacityShare": share,
        "Contacted": k,
        "CasesFound": found,
        "CapturedShare": found / cases,
        "PPV": found / k,
        "Lift": (found / k) / prevalence,
        "ContactedPer10k": round(10_000 * k / n),
        "CasesFoundPer10k": round(10_000 * found / n),
        "CasesFoundPer10kRandom": round(10_000 * k / n * prevalence),
        "Basis": "Analytic sample, unweighted (5,751 adults, prevalence 4.03%)",
    })
gains = pd.DataFrame(rows).round(4)
gains.to_csv(OUT_DIR / "gains_table.csv", index=False)
print(f"{n:,} adults, {cases} CHD cases, sample prevalence {prevalence:.2%}")
gains.drop(columns="Basis")

5,751 adults, 232 CHD cases, sample prevalence 4.03%


,CapacityShare,Contacted,CasesFound,CapturedShare,PPV,Lift,ContactedPer10k,CasesFoundPer10k,CasesFoundPer10kRandom
0,0.05,288,73,0.3147,0.2535,6.2833,501,127,20
1,0.10,576,116,0.5000,0.2014,4.9922,1002,202,40
2,0.20,1151,171,0.7371,0.1486,3.6828,2001,297,81
3,0.30,1726,199,0.8578,0.1153,2.8580,3001,346,121


## Reading the table

- Calling the riskiest **5%** (about 500 per 10,000 adults in this sample) reaches **31%** of CHD cases: 127 per 10,000 instead of 20 at random (PPV 25%, lift 6.3).
- **10%** reaches half of all cases (202 vs 40 per 10,000; PPV 20%, lift 5.0); **20%** reaches 74% (297 vs 81; lift 3.7); **30%** reaches 86% (346 vs 121; PPV 11.5%, lift 2.9), close to the 0.40-threshold operating point of the audit notebook (31% flagged, sensitivity 86%).
- Returns diminish: the first 10% of calls find half of the cases, the next 20% find another 36%.

All figures are unweighted and refer to the analytic sample. The model ranks people who already report a CHD diagnosis, so this illustrates how to size an outreach list against capacity, not how a deployed member-screening model would perform.
